# Calculate EU mortality with parametric bootstrapping

This may require large memory ~60GB.

In [1]:
import os
import xarray as xr
from utils.mortality_utils import mortality
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [3]:
# === Path config ===
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "population")
BMR_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "BMR")
RR_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "rr_pm25")

In [5]:
# === Calculate the scalar distributions ===
n_samples = 1000

In [10]:
# === Scenario and path config ===
# For RR curves and file name
GBD_version = "GBD23"

scenarios = ["H", "HL", "L", "LN", "M", "ML", "VL"]
years = [2040, 2060, 2080, 2100]

PM25_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "mortality" / "global" / f"{n_samples}_samples")

for health_VAR in health_vars:
    print(f"Processing health variable {health_VAR}")
    for scenario in scenarios:
        for year in years:
            print(f"Processing {scenario}, year {year}, {health_VAR}")

            # RR from GBD23 (normal distribution)
            rr_file = f"{GBD_version}_RR_{health_VAR}_{n_samples}_samples_pm25.nc"
            rr_path = os.path.join(RR_DIR, rr_file)
            rr_da = xr.open_dataarray(rr_path)

            del rr_file, rr_path

            # Load BMR for each grid point (normal distribution)
            bmr_file = f"{GBD_version}_BMR_Country_Mask_{health_VAR}_{n_samples}_samples_2015-2019.nc"
            bmr_path = os.path.join(BMR_DIR, bmr_file)
            BMR = xr.open_dataarray(bmr_path)

            del bmr_file, bmr_path

            pm25_file = f"EU_concentration_{scenario}_{year}.nc"
            pm25_path = os.path.join(PM25_DIR, pm25_file)
            pm25 = xr.open_dataarray(pm25_path)

            del pm25_file, pm25_path

            # Find the RR at each grid point
            RR = rr_da.interp(exposure=pm25)
            RR = RR.drop_vars(["exposure"])

            del pm25

            # Calculate the attributable fraction
            AF = (1 - (1/RR))

            del RR

            # Load population file
            pop_file = f"Population_count_regridded_{year}_{scenario}.nc"
            pop_path = os.path.join(POP_DIR, pop_file)
            POP = xr.open_dataarray(pop_path)

            # Calculate mortality at each grid point for n samples
            M = mortality(AF, BMR, POP)

            del AF, POP

            # Calculate the total global mortality
            EU_M = M.sum(dim=("latitude", "longitude"))

            del M

            description = (f"EU {health_VAR} mortality due to PM2.5 "
                           "- scripts by A.F. Wells (2025)")
            EU_M.attrs["description"] = description
            EU_M.attrs["GBD version"] = GBD_version
            EU_M.attrs["health_var"] = health_VAR
            EU_M.attrs["scenario"] = scenario
            EU_M.attrs["year"] = year

            out_file = f"EU_mortality_{GBD_version}_{health_VAR}_{n_samples}samples_{scenario}_{year}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)
            print(f"Saving to {out_path}")
            EU_M.to_netcdf(out_path)

    del rr_da, BMR

print("All processing complete.")

Processing health variable COPD
Processing H, year 2040, COPD
Saving to /glade/work/awells/EU_pm/mortality/global/1000_samples/EU_mortality_GBD23_COPD_1000samples_H_2040.nc
Processing H, year 2060, COPD
Saving to /glade/work/awells/EU_pm/mortality/global/1000_samples/EU_mortality_GBD23_COPD_1000samples_H_2060.nc
Processing H, year 2080, COPD
Saving to /glade/work/awells/EU_pm/mortality/global/1000_samples/EU_mortality_GBD23_COPD_1000samples_H_2080.nc
Processing H, year 2100, COPD
Saving to /glade/work/awells/EU_pm/mortality/global/1000_samples/EU_mortality_GBD23_COPD_1000samples_H_2100.nc
Processing HL, year 2040, COPD
Saving to /glade/work/awells/EU_pm/mortality/global/1000_samples/EU_mortality_GBD23_COPD_1000samples_HL_2040.nc
Processing HL, year 2060, COPD
Saving to /glade/work/awells/EU_pm/mortality/global/1000_samples/EU_mortality_GBD23_COPD_1000samples_HL_2060.nc
Processing HL, year 2080, COPD
Saving to /glade/work/awells/EU_pm/mortality/global/1000_samples/EU_mortality_GBD23_COP